In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 21
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.457628613813313

Trial 1 =========================================
13.887235069331375

Trial 2 =========================================
16.027391432854927

Trial 3 =========================================
14.171488942125672

Trial 4 =========================================
14.641715285314953

Trial 5 =========================================
13.268847944108273

Trial 6 =========================================
13.74584361548301

Trial 7 =========================================
17.094012498321096

Trial 8 =========================================
13.803800922934432

Trial 9 =========================================
13.778206226048201

Trial 10 =========================================
13.927581021137993

Trial 11 =========================================
13.882890141716251

Trial 12 =========================================
13.356314688415386

Trial 13 =========================================
16.18322940548244



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 14 =========================================
15.929222010399386

Trial 15 =========================================
14.848551325815064

Trial 16 =========================================
17.432974638029016

Trial 17 =========================================
14.843798066989153

Trial 18 =========================================
15.72399738847205

Trial 19 =========================================
18.599867880854944

Trial 20 =========================================
16.042814443288236

Trial 21 =========================================
17.56825112533675

Trial 22 =========================================
13.487285238729557

Trial 23 =========================================
17.48913245111913

Trial 24 =========================================
15.237282004251325

Trial 25 =========================================
14.227474753352691

Trial 26 =========================================
16.386505396240096

Trial 27 =========================================
16.181663226385332



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 28 =========================================
16.220289594485667

Trial 29 =========================================
15.245071863765148

Trial 30 =========================================
15.329106749757482

Trial 31 =========================================
14.46227342582418

Trial 32 =========================================
14.014686118197623

Trial 33 =========================================
13.941826548888502

Trial 34 =========================================
14.850281616642992

Trial 35 =========================================
14.464124475418222



...

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 79 =========================================
16.214180137187704

Trial 80 =========================================
15.667150564973483

Trial 81 =========================================
14.437158807328695

Trial 82 =========================================
16.0197511862626

Trial 83 =========================================
14.16200831142203

Trial 84 =========================================
15.07536218801533

Trial 85 =========================================
17.146231664102135

Trial 86 =========================================
14.650350091254005

Trial 87 =========================================
16.056950228991465

Trial 88 =========================================
13.55660217918517

Trial 89 =========================================
14.298902970793009

Trial 90 =========================================
14.388956276295172

Trial 91 =========================================
13.760541127673704

Trial 92 =========================================
16.071032395818996

Trial 93 ==

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 99 =========================================
15.533942467350526

[14.45762861 13.88723507 16.02739143 14.17148894 14.64171529 13.26884794
 13.74584362 17.0940125  13.80380092 13.77820623 13.92758102 13.88289014
 13.35631469 16.18322941 15.92922201 14.84855133 17.43297464 14.84379807
 15.72399739 18.59986788 16.04281444 17.56825113 13.48728524 17.48913245
 15.237282   14.22747475 16.3865054  16.18166323 16.22028959 15.24507186
 15.32910675 14.46227343 14.01468612 13.94182655 14.85028162 14.46412448
 16.098909   15.25247383 15.6617566  15.70476196 15.77332313 13.77650195
 13.99634987 14.00055354 14.44211885 14.18590751 16.08311813 13.60348931
 16.53919188 16.2088847  16.00387457 14.28090653 13.88856533 15.86406257
 16.24664145 17.07046009 13.22832659 14.27388238 15.3894003  14.85026601
 15.03992422 15.66024593 17.64246565 14.83270964 15.31252578 16.2318138
 14.48984324 15.93671417 15.92922201 13.92574308 15.26079498 15.61499119
 13.88294057 16.84621102 14.83959835 16.24324942 14.14

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.599867880854944
Avg = 15.124298402026323
Std = 1.168087765255876


In [6]:
print(y_max_arr.tolist())

[14.457628613813313, 13.887235069331375, 16.027391432854927, 14.171488942125672, 14.641715285314953, 13.268847944108273, 13.74584361548301, 17.094012498321096, 13.803800922934432, 13.778206226048201, 13.927581021137993, 13.882890141716251, 13.356314688415386, 16.18322940548244, 15.929222010399386, 14.848551325815064, 17.432974638029016, 14.843798066989153, 15.72399738847205, 18.599867880854944, 16.042814443288236, 17.56825112533675, 13.487285238729557, 17.48913245111913, 15.237282004251325, 14.227474753352691, 16.386505396240096, 16.181663226385332, 16.220289594485667, 15.245071863765148, 15.329106749757482, 14.46227342582418, 14.014686118197623, 13.941826548888502, 14.850281616642992, 14.464124475418222, 16.098909001757036, 15.25247383417015, 15.661756599596416, 15.704761962253215, 15.773323125780287, 13.776501953194238, 13.996349865312272, 14.000553543667989, 14.442118848728326, 14.18590751056612, 16.083118132147323, 13.603489308397963, 16.53919188120376, 16.20888469898162, 16.003874

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-21.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)